In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
import natural_units as nu
from scipy.special import erf
import scipy.integrate
from scipy.integrate import quad
from scipy.interpolate import RegularGridInterpolator
from scipy.interpolate import interp1d
import re
# multi-core/thread:
import concurrent.futures

In [2]:
# Unified original + extended grid.
log_m_min  = -3
log_m_max  = 1
n_m    = 33   # From 1e-3  to 10 GeV
# log10 cross-section shift from the balloon line
log_cs_min = -6
log_cs_max = 0
n_cs   = 97
m_grid  = np.logspace(log_m_min, log_m_max, n_m)
cs_shift_grid = np.linspace(log_cs_max, log_cs_min, n_cs)
center_line = np.array([2.15504637e-23, 1.88334907e-23, 1.53030461e-23, 1.32738030e-23,
       1.26359147e-23, 1.37889395e-23, 1.61669130e-23, 1.95514385e-23,
       2.40008514e-23, 3.14137096e-23, 4.03532201e-23, 5.19960700e-23,
       6.95747264e-23, 9.69011074e-23, 1.40957345e-22, 2.05768490e-22,
       2.84518575e-22, 3.82210762e-22, 5.17381239e-22, 7.29305911e-22,
       1.08351297e-21, 1.55439713e-21, 2.15203017e-21, 3.00211451e-21,
       4.12016417e-21, 5.67974728e-21, 7.78952222e-21, 1.06757849e-20,
       1.45615810e-20, 1.99263396e-20, 2.70914228e-20, 3.67600191e-20,
       4.94000000e-20])
atten_surv_ratio = np.zeros((n_cs,n_m))
signal_grid = np.zeros((n_cs,n_m))
atten_filled = np.zeros((n_cs,n_m), dtype=bool)
signal_filled = np.zeros((n_cs,n_m), dtype=bool)
pattern = re.compile(r"mDM=([0-9.eE+-]+)_GeV_sigma=([0-9.eE+-]+)_cm2")

def parameter_key(filename):
    match = pattern.search(filename.stem)
    if match is None:
        return None
    return float(match.group(1)), float(match.group(2))

def select_unique_files(path):
    selected = {}
    for filename in path.glob('*.txt'):
        key = parameter_key(filename)
        if key is None:
            continue
        if key in selected:
            raise RuntimeError(f'Duplicate physical point in {path}: {key}')
        selected[key] = filename
    return list(selected.values())

def grid_indices(filename):
    mDM_in_GeV, sigma_e_in_cm2 = parameter_key(filename)
    m_index = np.argmin(np.abs(np.log10(m_grid) - np.log10(mDM_in_GeV)))
    cs_shift = np.log10(sigma_e_in_cm2 / center_line[m_index])
    cs_index = np.argmin(np.abs(cs_shift_grid - cs_shift))
    if abs(np.log10(m_grid[m_index] / mDM_in_GeV)) > 1e-4 or abs(cs_shift_grid[cs_index] - cs_shift) > 1e-4:
        raise ValueError(f'{filename.name} does not lie on the expected 49 x 17 grid.')
    return cs_index, m_index

In [3]:
result_files = select_unique_files(Path('./result'))
for filename in result_files:
    cs_index, m_index = grid_indices(filename)
    signal_grid[cs_index, m_index] = np.loadtxt(filename)[2]
    signal_filled[cs_index, m_index] = True

speed_pdf_files = select_unique_files(Path('../data/DM_Speed_PDF'))
for filename in speed_pdf_files:
    cs_index, m_index = grid_indices(filename)
    speed_pdf = np.loadtxt(filename)
    atten_surv_ratio[cs_index, m_index] = scipy.integrate.simpson(speed_pdf[:,1], x=speed_pdf[:,0])
    atten_filled[cs_index, m_index] = True

if not np.all(signal_filled):
    raise RuntimeError(f'Missing {np.count_nonzero(~signal_filled)} signal-grid points.')
if not np.all(atten_filled):
    raise RuntimeError(f'Missing {np.count_nonzero(~atten_filled)} attenuation-grid points.')

# The only grid outputs are the unified 49-row arrays.
np.savetxt('./signal_total_grid.txt', signal_grid, fmt='%.3e')
np.savetxt('./atten_surv_ratio_total_grid.txt', atten_surv_ratio, fmt='%.3e')

print(f'Signal grid: {len(result_files)} unique points.')
print(f'Attenuation grid: {len(speed_pdf_files)} unique points.')

Signal grid: 3201 unique points.
Attenuation grid: 3201 unique points.
